# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.318542181540501)),
 (1, np.float64(2.318542181540501)),
 (2, np.float64(2.318542181540501)),
 (3, np.float64(2.318542181540501)),
 (4, np.float64(2.318542181540501))]

## Inverted index

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [('banana', 2)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.19909838204396868)),
   (None, np.float64(0.2000439180074718)),
   (None, np.float64(0.2134669279143353)),
   (None, np.float64(0.22460720171902404)),
   (None, np.float64(0.2724257202294309)),
   (None, np.float64(0.2951038618724846)),
   (None, np.float64(0.34405947074528265)),
   (None, np.float64(0.3586211134213614)),
   (None, np.float64(0.38393121181707013)),
   (None, np.float64(0.3995764145921227)),
   (None, np.float64(0.43532084797840764)),
   (None, np.float64(0.4476573745953222)),
   (None, np.float64(0.459437326714666)),
   (None, np.float64(0.47638028959822387)),
   (None, np.float64(0.48365182146395747))]),
 (1,
  [(None, np.float64(0.549947986963991)),
   (None, np.float64(0.5911176780433404)),
   (None, np.float64(0.6041213317169901)),
   (None, np.float64(0.6407911230900174)),
   (None, np.float64(0.662095657415228)),
   (None, np.float64(0.683830579139954)),
   (None, np.float64(0.688420997474399)),
   (None, np.float64(0.7427539516286399)

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
def RECORDREADER():
    # Пример списка чисел
    numbers = [3, 7, 2, 9, 5, 1, 8]
    return [(i, num) for i, num in enumerate(numbers)]

def MAP(_, num):
    yield (num, None)   # ключ = число, значение не используется

def REDUCE(key, values):
    # ключ — это само число, просто передаём его дальше
    yield (key, None)

# Запуск MapReduce
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max_value = max(k for k, _ in output)
print("Максимальное значение:", max_value)

Максимальное значение: 9


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
def RECORDREADER():
    numbers = [10, 20, 30, 40, 50]
    return [(i, num) for i, num in enumerate(numbers)]

def MAP(_, num):
    # Отправляем (1, (сумма, счётчик)) для глобальной агрегации
    yield (1, (num, 1))

def REDUCE(_, values):
    total_sum = 0
    total_count = 0
    for s, c in values:
        total_sum += s
        total_count += c
    mean = total_sum / total_count
    yield ("mean", mean)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('mean', 30.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [23]:
def groupbykey_sorted(iterable):
    # Преобразуем в список и сортируем по ключу
    items = sorted(iterable, key=lambda x: x[0])
    result = []
    current_key = None
    current_values = []
    for k, v in items:
        if k != current_key:
            if current_key is not None:
                result.append((current_key, current_values))
            current_key = k
            current_values = [v]
        else:
            current_values.append(v)
    if current_key is not None:
        result.append((current_key, current_values))
    return result

# Тест на примере
test_data = [('a', 1), ('b', 2), ('a', 3), ('c', 4), ('b', 5)]
grouped = groupbykey_sorted(test_data)
print(grouped)

[('a', [1, 3]), ('b', [2, 5]), ('c', [4])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [24]:
def RECORDREADER():
    # Пример с дубликатами
    data = [1, 2, 2, 3, 4, 4, 4, 5]
    return [(i, x) for i, x in enumerate(data)]

def MAP(_, value):
    yield (value, None)   # ключ = элемент, значение игнорируется

def REDUCE(key, values):
    # Отдаём каждый ключ только один раз
    yield (key, None)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
unique = [k for k, _ in output]
print(unique)

[1, 2, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [25]:
def RECORDREADER():
    # Пример кортежей: (id, name, age)
    data = [
        (1, 'Alice', 25),
        (2, 'Bob', 30),
        (3, 'Charlie', 35),
        (4, 'Diana', 25)
    ]
    return [(i, t) for i, t in enumerate(data)]

def MAP(_, t):
    # Предикат: возраст == 25
    if t[2] == 25:
        yield (t, t)

def REDUCE(key, values):
    # Тождественная функция
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
selected = [t for t, _ in output]
print(selected)

[(1, 'Alice', 25), (4, 'Diana', 25)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [26]:
def RECORDREADER():
    # (id, name, age, salary)
    data = [
        (1, 'Alice', 25, 50000),
        (2, 'Bob', 30, 60000),
        (3, 'Charlie', 35, 70000)
    ]
    return [(i, t) for i, t in enumerate(data)]

def MAP(_, t):
    # Проекция на (name, age) — индексы 1 и 2
    projected = (t[1], t[2])
    yield (projected, projected)

def REDUCE(key, values):
    # Оставляем только одну копию
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
result = [v for _, v in output]
print(result)

[('Alice', 25), ('Bob', 30), ('Charlie', 35)]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [27]:
def RECORDREADER():
    # Отношение R
    R = [(1, 'A'), (2, 'B')]
    # Отношение S
    S = [(2, 'B'), (3, 'C')]
    # Объединяем списки
    records = [(i, t) for i, t in enumerate(R + S)]
    return records

def MAP(_, t):
    yield (t, t)

def REDUCE(key, values):
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
union = [t for t, _ in output]
print(union)

[(1, 'A'), (2, 'B'), (3, 'C')]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [28]:
def RECORDREADER():
    # Метим каждый кортеж номером отношения: 0 для R, 1 для S
    R = [(1, 'A'), (2, 'B'), (4, 'D')]
    S = [(2, 'B'), (3, 'C'), (4, 'D')]
    records = []
    for t in R:
        records.append((len(records), (t, 0)))
    for t in S:
        records.append((len(records), (t, 1)))
    return records

def MAP(_, tagged_tuple):
    t, rel_id = tagged_tuple
    yield (t, rel_id)

def REDUCE(key, values):
    # Если есть оба rel_id 0 и 1, значит кортеж есть в обоих отношениях
    rel_set = set(values)
    if 0 in rel_set and 1 in rel_set:
        yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
intersection = [t for t, _ in output]
print(intersection)

[(2, 'B'), (4, 'D')]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [29]:
def RECORDREADER():
    R = [(1, 'A'), (2, 'B'), (4, 'D')]
    S = [(2, 'B'), (3, 'C')]
    records = []
    for t in R:
        records.append((len(records), (t, 'R')))
    for t in S:
        records.append((len(records), (t, 'S')))
    return records

def MAP(_, tagged_tuple):
    t, rel = tagged_tuple
    yield (t, rel)

def REDUCE(key, values):
    # Если есть только 'R' — сохраняем кортеж
    if list(values) == ['R']:
        yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
difference = [t for t, _ in output]
print(difference)

[(1, 'A'), (4, 'D')]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [30]:
def RECORDREADER():
    # R: (a,b)
    R = [(1, 2), (3, 4), (5, 6)]
    # S: (b,c)
    S = [(2, 10), (4, 20), (7, 30)]
    records = []
    for a,b in R:
        records.append((len(records), (b, ('R', a))))
    for b,c in S:
        records.append((len(records), (b, ('S', c))))
    return records

def MAP(_, pair):
    b, tagged = pair
    yield (b, tagged)

def REDUCE(key, values):
    # values — список ('R', a) или ('S', c)
    r_vals = [a for rel, a in values if rel == 'R']
    s_vals = [c for rel, c in values if rel == 'S']
    for a in r_vals:
        for c in s_vals:
            yield ((a, key, c), None)   # выдаём тройку (a,b,c)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
result = [t for t, _ in output]
print(result)

[(1, 2, 10), (3, 4, 20)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [31]:
def RECORDREADER():
    # (ключ группы, значение)
    data = [('A', 10), ('B', 20), ('A', 30), ('B', 5), ('A', 15)]
    return [(i, row) for i, row in enumerate(data)]

def MAP(_, row):
    key, val = row
    yield (key, val)

def REDUCE(key, values):
    # Агрегация: сумма
    total = sum(values)
    yield (key, total)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('A', 55), ('B', 25)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [33]:
import numpy as np

matrix = np.random.rand(100, 100)
vector = np.random.rand(100)

maps = 4
reducers = 1

def INPUTFORMAT():
    rows_per_split = int(np.ceil(matrix.shape[0] / maps))
    for i in range(0, matrix.shape[0], rows_per_split):
        rows_slice = slice(i, min(i+rows_per_split, matrix.shape[0]))
        def record_reader():
            for r in range(rows_slice.start, rows_slice.stop):
                for c in range(matrix.shape[1]):
                    yield ((r, c), matrix[r, c])
        yield record_reader()

def MAP(key, value):
    r, c = key
    m_val = value
    yield (r, m_val * vector[c])

def REDUCE(key, values):
    total = sum(values)
    yield (key, total)

reducers = 1
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
result = {k: v for (_, part) in partitioned_output for (k, v) in part}
print("Результат (первые 5 строк):", list(result.items())[:5])

10000 key-value pairs were sent over a network.
Результат (первые 5 строк): [(0, np.float64(24.46722161748208)), (1, np.float64(21.710680410598485)), (2, np.float64(23.79694450145329)), (3, np.float64(20.704797636424754)), (4, np.float64(22.61541552436046))]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [34]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [38]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):          # I = 2
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  total = sum(values)
  yield ((i, k), total)

Проверьте своё решение

In [39]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [40]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [41]:
import numpy as np

I = 2
J = 3
K = 4 * 10
small_mat = np.random.rand(I, J)   # в памяти
big_mat = np.random.rand(J, K)

def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
    j, k = k1
    w = v1
    # Для каждой строки i матрицы small_mat вычисляем частичное произведение
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
    i, k = key
    total = sum(values)
    yield ((i, k), total)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Первые 5 результатов:", output[:5])

Первые 5 результатов: [((0, 0), np.float64(0.21379986305770843)), ((1, 0), np.float64(0.6299846961441862)), ((0, 1), np.float64(0.6389271408730745)), ((1, 1), np.float64(0.8303154483016761)), ((0, 2), np.float64(0.21046359517525232))]


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [42]:
import numpy as np

I, J, K = 3, 4, 5
matM = np.random.rand(I, J)
matN = np.random.rand(J, K)

def RECORDREADER():
    # Сначала выдаём элементы M с меткой 'M'
    for i in range(I):
        for j in range(J):
            yield ((i, j, 'M'), matM[i, j])
    # Затем элементы N с меткой 'N'
    for j in range(J):
        for k in range(K):
            yield ((j, k, 'N'), matN[j, k])

def MAP(key, value):
    if key[2] == 'M':   # M(i,j)
        i, j, _ = key
        yield (j, ('M', i, value))
    else:               # N(j,k)
        j, k, _ = key
        yield (j, ('N', k, value))

def REDUCE2(key, values):
    # key = j, values = список ('M',i,v) и ('N',k,v)
    m_dict = {}
    n_dict = {}
    for item in values:
        if item[0] == 'M':
            _, i, v = item
            m_dict[i] = v
        else:
            _, k, v = item
            n_dict[k] = v
    for i, mval in m_dict.items():
        for k, nval in n_dict.items():
            yield ((i, k), mval * nval)

# Первая стадия: соединение по j и умножение
intermediate = list(MapReduce(RECORDREADER, MAP, REDUCE2))

# Вторая стадия: суммирование по одинаковым (i,k)
def final_RECORDREADER():
    for ((i,k), prod) in intermediate:
        yield ((i,k), prod)

def final_MAP(key, prod):
    yield (key, prod)

def final_REDUCE(key, values):
    yield (key, sum(values))

result = list(MapReduce(final_RECORDREADER, final_MAP, final_REDUCE))
print("Результат перемножения матриц (первые 5):", result[:5])

Результат перемножения матриц (первые 5): [((0, 0), np.float64(1.5985145137309638)), ((0, 1), np.float64(0.6740981884161317)), ((0, 2), np.float64(1.522714601580787)), ((0, 3), np.float64(1.4900581910635955)), ((0, 4), np.float64(1.0493190978354903))]


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [44]:
import numpy as np

I, J, K = 10, 20, 30
matM = np.random.rand(I, J)
matN = np.random.rand(J, K)

maps = 3
reducers = 2

def INPUTFORMAT():
    # Разбиваем элементы M на части
    all_m_entries = [((i,j,'M'), matM[i,j]) for i in range(I) for j in range(J)]
    split_size = int(np.ceil(len(all_m_entries) / maps))
    for start in range(0, len(all_m_entries), split_size):
        split = all_m_entries[start:start+split_size]
        def reader():
            for item in split:
                yield item
        yield reader()
    # Аналогично для N
    all_n_entries = [((j,k,'N'), matN[j,k]) for j in range(J) for k in range(K)]
    for start in range(0, len(all_n_entries), split_size):
        split = all_n_entries[start:start+split_size]
        def reader():
            for item in split:
                yield item
        yield reader()

def MAP(key, value):
    if key[2] == 'M':
        _, j, _ = key
        yield (j, ('M', key[0], value))
    else:
        j, k, _ = key
        yield (j, ('N', k, value))

def REDUCE(key, values):
    m_vals = [(i, v) for (tag, i, v) in values if tag == 'M']
    n_vals = [(k, v) for (tag, k, v) in values if tag == 'N']
    for (i, mval) in m_vals:
        for (k, nval) in n_vals:
            yield ((i, k), mval * nval)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=lambda x: hash(x) % reducers, COMBINER=None)

# Собираем все произведения
all_products = []
for _, part in partitioned_output:
    all_products.extend(list(part))

# Вторая стадия: суммирование по (i,k)
def second_RECORDREADER():
    for ((i,k), prod) in all_products:
        yield ((i,k), prod)

def second_MAP(key, prod):
    yield (key, prod)

def second_REDUCE(key, values):
    yield (key, sum(values))

final_result = list(MapReduce(second_RECORDREADER, second_MAP, second_REDUCE))
print("Финальное произведение матриц (первые 5 записей):", final_result[:5])

800 key-value pairs were sent over a network.
Финальное произведение матриц (первые 5 записей): [((0, 0), np.float64(2.4404111707167924)), ((0, 1), np.float64(3.7780619607339734)), ((0, 2), np.float64(4.215997132856926)), ((0, 3), np.float64(4.255418628806002)), ((0, 4), np.float64(4.047780147462576))]
